<a href="https://colab.research.google.com/github/vishakha19-hue/vishakha/blob/main/irisflowerclassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas matplotlib seaborn scikit-learn numpy


In [4]:
"""
CodeAlpha Data Science Internship
Task 1: Iris Flower Classification
------------------------------------------------------------
Goal: Predict the species of an Iris flower (Setosa, Versicolor,
Virginica) based on its measurements (sepal length, sepal width,
petal length, petal width).
"""

# ---------- STEP 1: Import required libraries ----------
import pandas as pd                     # to handle data in table (DataFrame) format
import matplotlib.pyplot as plt         # to create graphs/plots
import seaborn as sns                   # for nicer-looking graphs than plain matplotlib

from sklearn.datasets import load_iris                  # built-in iris dataset
from sklearn.model_selection import train_test_split    # to split data into train/test sets
from sklearn.preprocessing import StandardScaler        # to bring features to the same scale
from sklearn.neighbors import KNeighborsClassifier       # KNN classification model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# ---------- STEP 2: Load the dataset ----------
iris = load_iris()

# iris.data contains the measurements (numbers), iris.target contains the species (0,1,2)
X = pd.DataFrame(iris.data, columns=iris.feature_names)   # X = input features
y = pd.Series(iris.target, name="species")                # y = output (what we want to predict)

# Convert target numbers (0,1,2) into actual names (setosa, versicolor, virginica)
# so the graphs and output are easier to understand
species_map = {0: "setosa", 1: "versicolor", 2: "virginica"}
y_named = y.map(species_map)

print("Dataset shape (rows, columns):", X.shape)
print("\nFirst 5 rows:\n", X.head())
print("\nSample count per species:\n", y_named.value_counts())


# ---------- STEP 3: Exploratory Data Analysis (EDA) ----------
# A pairplot helps us see how different species differ from (or resemble)
# each other in terms of their measurements
df_plot = X.copy()
df_plot["species"] = y_named

sns.pairplot(df_plot, hue="species")
plt.suptitle("Comparison of Iris Species Measurements", y=1.02)
plt.savefig("iris_pairplot.png", bbox_inches="tight")
plt.close()
print("\n'iris_pairplot.png' saved - it shows that the species form distinct clusters.")


# ---------- STEP 4: Split data into Train and Test sets ----------
# 80% of the data is used to train the model, 20% is kept aside for testing (unseen data)
# random_state=42 ensures we get the same split every time we run the code (reproducibility)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining samples: {len(X_train)}, Testing samples: {len(X_test)}")


# ---------- STEP 5: Feature Scaling ----------
# Distance-based algorithms like KNN need all features on the same scale,
# otherwise a feature with a larger range would dominate the result
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn mean/std from training data and apply it
X_test_scaled = scaler.transform(X_test)         # only apply (don't re-learn) on test data, to avoid data leakage


# ---------- STEP 6: Train the model (KNN Classifier) ----------
# KNN (K-Nearest Neighbors): to predict a new point, it looks at the
# "k" closest training points and predicts whichever species is
# most common among them
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)


# ---------- STEP 7: Make predictions and check accuracy ----------
y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy * 100:.2f}%")

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=species_map.values()))

# The Confusion Matrix shows us where the model predicts correctly and where it makes mistakes
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=species_map.values(), yticklabels=species_map.values())
plt.xlabel("Predicted Species")
plt.ylabel("Actual Species")
plt.title("Confusion Matrix - Iris Classification")
plt.savefig("iris_confusion_matrix.png", bbox_inches="tight")
plt.close()
print("'iris_confusion_matrix.png' saved.")


# ---------- STEP 8: Predict a new flower as an example ----------
# Let's take a new sample and see what the model predicts
sample_flower = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],   # sepal_length, sepal_width, petal_length, petal_width
    columns=iris.feature_names
)
sample_scaled = scaler.transform(sample_flower)
prediction = model.predict(sample_scaled)[0]
print(f"\nExample prediction -> New flower ({sample_flower.values.tolist()[0]}) predicted species: {species_map[prediction]}")

print("\n✅ Task 1 (Iris Flower Classification) complete!")

Dataset shape (rows, columns): (150, 4)

First 5 rows:
    sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                4.9               3.0                1.4               0.2
2                4.7               3.2                1.3               0.2
3                4.6               3.1                1.5               0.2
4                5.0               3.6                1.4               0.2

Sample count per species:
 species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64

'iris_pairplot.png' saved - it shows that the species form distinct clusters.

Training samples: 120, Testing samples: 30

Model Accuracy: 93.33%

Detailed Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.83      1.00      0.91        10
   virginica       1.00      0.8